# Logistic regression, Extra Trees, and LightGBM on frozen BERT embeddings

Embeddings are frozen features. The 30 stratified splits are taken on the
deduplicated union of the binary files. A prompt that occurred in both the old
train and test files is kept once, so it cannot be fit and scored in the same run.

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

import numpy as np
import torch
from tqdm import tqdm
from transformers import BertModel, BertTokenizer

from binary_data import load_clean_binary_corpus
from classical_experiments import run_matrix_model_runs
from latex_tables import export_results

NUM_RUNS = int(os.environ.get("NUM_RUNS", "30"))
corpus, report = load_clean_binary_corpus()
print(f"Corpus rows={len(corpus)} shared_prompts_removed={report['shared_prompts_before']}")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")
bert_model.eval()

def extract_bert_features(texts, tokenizer, model, max_length=128):
    embeddings = []
    for text in tqdm(texts, desc="Extracting BERT features"):
        tokens = tokenizer.encode_plus(
            str(text),
            max_length=max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            output = model(input_ids=tokens["input_ids"], attention_mask=tokens["attention_mask"])
        embeddings.append(output.last_hidden_state[:, 0, :].squeeze().numpy())
    return np.vstack(embeddings)

features = extract_bert_features(corpus["prompt"], tokenizer, bert_model)
runs = run_matrix_model_runs(
    features,
    corpus["prompt"],
    corpus["jailbreak"],
    features_name="BERT",
    num_runs=NUM_RUNS,
    max_iter=1000,
)
paths = export_results("04_bert_features", "binary", runs)
print(f"Wrote {paths['tex']}")